# Module 2.1 -- High-Performance Async Serving & LLM Token Streaming

**Track:** Backend Engineering for AI Applications  
**Toolchain:** FastAPI - Litestar - Server-Sent Events (SSE) - ASGI  
**Objective:** Build production async APIs that stream LLM tokens to clients with sub-second time-to-first-token.

---

## Why AI Backends Are Different from Traditional APIs

Traditional API: Client sends request -> Server responds in 50ms -> Done.  
LLM API: Client sends request -> Server starts generating -> Tokens arrive over 3-30 SECONDS.

**If you wait for the full response:** User stares at a blank screen for 10 seconds.  
**If you stream tokens:** User sees words appear in real-time (like ChatGPT).

### Streaming Protocol Comparison

| Protocol | Direction | Complexity | Best For |
|----------|-----------|-----------|----------|
| **SSE** | Server -> Client only | Low | LLM token streaming (the standard) |
| **WebSocket** | Bidirectional | High | Chat apps, gaming |
| **gRPC streaming** | Bidirectional | High | Internal microservices |
| **Long polling** | Simulated streaming | Medium | Legacy browser support |

### Why SSE Won for LLM Streaming

1. **HTTP-native**: works with every load balancer, CDN, and firewall
2. **Auto-reconnect**: browser reconnects automatically if connection drops
3. **Simple**: just `text/event-stream` content type, no special protocol
4. **OpenAI standard**: the entire LLM industry adopted SSE via OpenAI's API


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** LLMs are slow. If it takes 15 seconds to generate an answer, a synchronous API will cause the user to think the app froze. Seniors use async HTTP and Server-Sent Events (SSE) to stream tokens instantly.

**Mental Model:** Synchronous serving is waiting for the entire car to be built before driving. Streaming (SSE) is building the car around you while driving it—showing the user one word at a time.

**Common Junior Pitfall:** Running a heavy PyTorch CPU blocking task inside an `async def` FastAPI route, which freezes the entire event loop and causes the whole API server to stall for all other users.

---


---

## Syntax Bridge: async/await in 60 Seconds

If `async/await` is new to you, here's the mental model (covered in detail in Notebook 00):

- **`async def`** = This function can PAUSE and let others run
- **`await`** = PAUSE HERE until the result comes back
- **Why?** Without async, your server handles 1 user at a time. With async, it handles 1000s.

```python
# Sync: server BLOCKS on each LLM call (1 user at a time)
def handle(request):
    result = call_llm(request)    # BLOCKS for 5 seconds
    return result

# Async: server handles many users while waiting (1000s concurrent)
async def handle(request):
    result = await call_llm(request)  # PAUSES, lets others run
    return result
```

The rest of this notebook uses `async` patterns throughout. If anything feels confusing, revisit Notebook 00 Section 3.


In [ ]:
# Cell 1 -- Install
!pip install -q fastapi uvicorn httpx sse-starlette pydantic


---
## 1 - FastAPI vs Litestar: Framework Comparison

| Feature | FastAPI | Litestar |
|---------|---------|----------|
| Validation | Pydantic (feature-rich) | msgspec (2-5x faster) |
| DI model | Depends() decorator | Testing-inspired, explicit |
| Performance | Fast | Faster (less overhead) |
| Ecosystem | Massive (most popular) | Growing |
| OpenAPI docs | Auto-generated | Auto-generated |
| Best for | Most projects | High-throughput inference APIs |

### When to Choose Litestar Over FastAPI?

- Inference APIs serving >10K req/sec (msgspec validation is faster)
- When you need strict DI patterns (Litestar's DI is more testable)
- When JSON serialization is a bottleneck

For most projects, **FastAPI is the safe default**. Choose Litestar when perf benchmarks show FastAPI is the bottleneck.


In [ ]:
# Cell 2 -- FastAPI inference endpoint with Pydantic validation
from pydantic import BaseModel, Field
from typing import List, Optional
import time, json

# Pydantic models enforce type safety at API boundary
class CompletionRequest(BaseModel):
    model: str = Field(default='gpt-4o', description='Model to use')
    messages: List[dict] = Field(..., min_length=1, description='Chat messages')
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    max_tokens: int = Field(default=256, ge=1, le=4096)
    stream: bool = Field(default=False, description='Enable SSE streaming')

class CompletionResponse(BaseModel):
    id: str
    model: str
    content: str
    usage: dict
    latency_ms: float

# Demonstrate validation
valid_req = CompletionRequest(messages=[{'role': 'user', 'content': 'Hello'}])
print(f'Valid request: {valid_req.model_dump_json(indent=2)}')

try:
    bad_req = CompletionRequest(messages=[], temperature=5.0)  # Both invalid!
except Exception as e:
    print(f'\nInvalid request caught by Pydantic: {e}')
    print('This validation happens BEFORE your model code runs -- protecting against bad inputs.')


In [ ]:
# Cell 3 -- SSE token streaming implementation
#
# This is the CORE pattern for streaming LLM tokens.
# Every word is sent as a separate SSE event as it's generated.

import asyncio

async def generate_tokens_sse(prompt: str, max_tokens: int = 50):
    """Simulate LLM token generation with SSE format."""
    response_words = f'Based on your question about {prompt[:20]}, here is a detailed analysis of the topic with relevant examples and practical recommendations for implementation.'.split()

    for i, token in enumerate(response_words[:max_tokens]):
        # SSE format: 'data: {json}\n\n' (double newline is REQUIRED)
        chunk = {
            'id': f'chatcmpl-{i}',
            'object': 'chat.completion.chunk',
            'choices': [{'index': 0, 'delta': {'content': token + ' '}, 'finish_reason': None}],
        }
        yield f'data: {json.dumps(chunk)}\n\n'
        await asyncio.sleep(0.05)  # Simulate generation latency

    # Final chunk signals completion
    yield f'data: {json.dumps({"choices": [{"finish_reason": "stop"}]})}\n\n'
    yield 'data: [DONE]\n\n'

# Demonstrate SSE output format
print('SSE Token Stream (what the client receives):')
print('=' * 60)
async def demo():
    async for chunk in generate_tokens_sse('machine learning', 8):
        print(chunk.strip())
asyncio.get_event_loop().run_until_complete(demo())


In [ ]:
# Cell 4 -- Full FastAPI app with streaming endpoint
fastapi_app = '''
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from sse_starlette.sse import EventSourceResponse
from pydantic import BaseModel, Field
from typing import List
import asyncio, json, time, uuid

app = FastAPI(title="LLM Inference API", version="2.0")

class CompletionRequest(BaseModel):
    messages: List[dict]
    stream: bool = False
    max_tokens: int = Field(default=256, ge=1, le=4096)

async def token_generator(prompt: str, max_tokens: int):
    words = f"Here is a comprehensive response to your query about {prompt}".split()
    for word in words[:max_tokens]:
        chunk = {"choices": [{"delta": {"content": word + " "}}]}
        yield {"data": json.dumps(chunk)}
        await asyncio.sleep(0.03)
    yield {"data": "[DONE]"}

@app.post("/v1/chat/completions")
async def chat_completions(req: CompletionRequest):
    prompt = req.messages[-1].get("content", "")
    if req.stream:
        return EventSourceResponse(token_generator(prompt, req.max_tokens))
    # Non-streaming: wait for full response
    full_text = f"Complete response to: {prompt}"
    return {"id": str(uuid.uuid4()), "content": full_text}

@app.get("/health")
def health(): return {"status": "healthy"}
'''
with open('llm_api_server.py', 'w') as f: f.write(fastapi_app)
print('FastAPI LLM Server written to llm_api_server.py')
print('Run with: uvicorn llm_api_server:app --host 0.0.0.0 --port 8000')
print('Docs at: http://localhost:8000/docs')


In [ ]:
# Cell 5 -- Async concurrency demonstration
# Shows WHY async matters: handling multiple requests simultaneously
import asyncio, time

async def simulate_llm_call(user_id, tokens=10):
    """Simulate an LLM generating tokens (I/O bound)."""
    start = time.perf_counter()
    for i in range(tokens):
        await asyncio.sleep(0.05)  # Simulates waiting for GPU
    elapsed = time.perf_counter() - start
    return f'User {user_id}: {tokens} tokens in {elapsed:.2f}s'

async def demo_sequential():
    start = time.perf_counter()
    results = []
    for i in range(5):
        results.append(await simulate_llm_call(i))
    return time.perf_counter() - start, results

async def demo_concurrent():
    start = time.perf_counter()
    tasks = [simulate_llm_call(i) for i in range(5)]
    results = await asyncio.gather(*tasks)
    return time.perf_counter() - start, results

async def main():
    seq_time, seq_res = await demo_sequential()
    con_time, con_res = await demo_concurrent()
    print('Sequential (sync-like): 1 user at a time')
    for r in seq_res: print(f'  {r}')
    print(f'  Total: {seq_time:.2f}s\n')
    print('Concurrent (async): all users simultaneously')
    for r in con_res: print(f'  {r}')
    print(f'  Total: {con_time:.2f}s')
    print(f'\nSpeedup: {seq_time/con_time:.1f}x faster with async!')

asyncio.get_event_loop().run_until_complete(main())


---
## 2 - Graceful Disconnect Handling

### The GPU Waste Problem

User sends a request, then closes the browser. Without disconnect detection, the GPU keeps generating tokens that nobody will receive. At $2/hour per GPU, this adds up fast.

```python
# MUST check if client is still connected
async def generate_with_cancel(request: Request, ...):
    for token in model.generate(...):
        if await request.is_disconnected():
            logger.info('Client disconnected, stopping generation')
            break  # Stop GPU work immediately
        yield format_sse(token)
```

### Why This Is Critical for Cost Control

| Scenario | Without disconnect check | With check |
|----------|-------------------------|------------|
| 1000 abandoned requests/day | 1000 x 30s GPU = 8.3 GPU-hours wasted | 1000 x 0.5s = 0.14 GPU-hours |
| Monthly cost @$2/hr | $500 wasted | $8 wasted |


In [ ]:
# Cell -- Graceful disconnect handling simulation
import asyncio

class MockRequest:
    def __init__(self, disconnect_after=5):
        self.token_count = 0
        self.disconnect_after = disconnect_after
    async def is_disconnected(self):
        return self.token_count >= self.disconnect_after

async def generate_with_cancel(request, max_tokens=20):
    tokens_sent = 0
    for i in range(max_tokens):
        if await request.is_disconnected():
            print(f'  Client disconnected after {tokens_sent} tokens. Stopping GPU.')
            print(f'  Saved {(max_tokens - tokens_sent) * 50}ms of GPU time!')
            return tokens_sent
        request.token_count += 1
        tokens_sent += 1
        await asyncio.sleep(0.01)
    print(f'  Completed: {tokens_sent} tokens generated')
    return tokens_sent

async def demo():
    print('Scenario 1: User stays connected')
    await generate_with_cancel(MockRequest(disconnect_after=100), max_tokens=10)
    print('\nScenario 2: User closes browser after 5 tokens')
    await generate_with_cancel(MockRequest(disconnect_after=5), max_tokens=20)

asyncio.get_event_loop().run_until_complete(demo())


---
## Knowledge Check

### Q1: Event Loop Blocking
A developer puts `time.sleep(5)` inside an `async def` FastAPI route. What happens to other users?

<details><summary>Answer</summary>

The **entire event loop freezes** for 5 seconds. ALL other users' requests are blocked. `time.sleep()` is a synchronous/blocking call. In async code, use `await asyncio.sleep(5)` instead, which pauses only that coroutine while others continue.
</details>

### Q2: SSE vs WebSocket
Why does the LLM industry use SSE instead of WebSockets for token streaming?

<details><summary>Answer</summary>

SSE is simpler (HTTP-native, works with all load balancers/CDNs), auto-reconnects on disconnect, and LLM streaming is server-to-client only (no bidirectional needed). WebSockets add unnecessary complexity for this use case.
</details>

### Q3: Disconnect Economics
Your API handles 50K requests/day, 10% are abandoned. Average generation is 30 seconds. How much GPU time do you waste per day without disconnect detection?

<details><summary>Answer</summary>

5,000 abandoned x 30s = 150,000 seconds = **41.7 GPU-hours/day** wasted. At $2/hr = $83/day = **$2,500/month**. With disconnect detection (avg 1s before catching): 5,000 x 1s = 1.4 GPU-hours = $2.8/day.
</details>


---
## Practical Practice

### Exercise 1: Rate Limiter
Write an async rate limiter that allows max 10 requests per second per user. Use `asyncio.Semaphore`.

### Exercise 2: SSE Format
Write a function that converts a list of tokens into proper SSE format (data: {json}\n\n).


In [ ]:
# ===== SOLUTIONS =====
import asyncio, json

print('Ex 1: Async Rate Limiter')
class RateLimiter:
    def __init__(self, max_per_sec=10):
        self.semaphore = asyncio.Semaphore(max_per_sec)
    async def acquire(self):
        await self.semaphore.acquire()
        asyncio.get_event_loop().call_later(1.0, self.semaphore.release)

limiter = RateLimiter(max_per_sec=10)
print('  RateLimiter(10/sec): uses Semaphore + timed release')

print('\nEx 2: SSE Formatter')
def tokens_to_sse(tokens):
    events = []
    for i, t in enumerate(tokens):
        chunk = {'choices': [{'delta': {'content': t}}]}
        events.append(f'data: {json.dumps(chunk)}\n\n')
    events.append('data: [DONE]\n\n')
    return events

for e in tokens_to_sse(['Hello', ' world', '!']):
    print(f'  {e.strip()}')


---
## Summary

| Concept | What You Learned |
|---------|------------------|
| Async frameworks | FastAPI (default) vs Litestar (high-perf) |
| SSE streaming | Industry standard for LLM token delivery |
| Pydantic validation | Type safety at API boundary |
| Disconnect handling | Stop GPU waste on abandoned requests |

**Next -->** `11_model_deployment.ipynb` -- Deploying Models with BentoML, vLLM & K8s